In [1]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 55.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import ultralytics
ultralytics.checks()  # confirms GPU is detected

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [4]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os
import shutil
import random
import numpy as np

# confirm GPU
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [7]:
data_yaml_path = f"{dataset.location}/data.yaml"
print("Dataset path:", dataset.location)
print("YAML file:", data_yaml_path)

Dataset path: /content/cvcep-3
YAML file: /content/cvcep-3/data.yaml


In [8]:
model = YOLO("yolov8m.pt")

# 'n' = nano, smallest and fastest
# yolov8s.pt (small), yolov8m.pt (medium)

In [9]:
results = model.train(
    data=data_yaml_path,
    epochs=100,
    patience=15,
    imgsz=640,
    batch=12,
    device=0,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # Reduced augmentation (you already used Roboflow)
    augment=True,
    degrees=5,
    translate=0.05,
    scale=0.3,
    shear=2,
    mosaic=0.5,
    close_mosaic=10,

    cache=True,
    workers=2,
    amp=True,

    project="/content/runs",
    name="blind_assist",
    exist_ok=True,
    save=True,
    val=True,
    plots=True,
    seed=SEED
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cvcep-3/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=blind_assist, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=15, p

In [10]:
best_path = None

for root, dirs, files in os.walk("/content/runs"):
    if "best.pt" in files:
        best_path = os.path.join(root, "best.pt")
        break

if best_path is None:
    raise FileNotFoundError("best.pt not found. Training may have failed.")

print("Best model found at:")
print(best_path)

Best model found at:
/content/runs/blind_assist/weights/best.pt


In [11]:
drive_save_path = "/content/drive/MyDrive/FYP/blind_assist_v2_best.pt"

shutil.copy(best_path, drive_save_path)

print("\n Model saved to Google Drive:",drive_save_path)



 Model saved to Google Drive: /content/drive/MyDrive/FYP/blind_assist_v2_best.pt


In [12]:
model = YOLO(drive_save_path)

print("\n Model loaded successfully from Drive")
print("Classes:", model.names)
names = model.names  # dict: {0: class_name}
data_yaml = data_yaml_path


 Model loaded successfully from Drive
Classes: {0: 'bench', 1: 'bin', 2: 'chair', 3: 'crosswalk', 4: 'dog', 5: 'door', 6: 'footpath', 7: 'person', 8: 'pole', 9: 'pothole', 10: 'stairs', 11: 'stopsign', 12: 'table', 13: 'trafficlight', 14: 'vehicle'}


In [13]:
val_metrics = model.val(data=data_yaml, split="val")

print("VALIDATION RESULTS")

print(f"mAP@0.5       : {val_metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {val_metrics.box.map:.3f}")
print(f"Precision     : {val_metrics.box.mp:.3f}")
print(f"Recall        : {val_metrics.box.mr:.3f}")

print("\n Per-Class Validation Performance:")

val_table = []

for i in range(len(names)):
    name = names[i]
    p = val_metrics.box.p[i]
    r = val_metrics.box.r[i]
    ap50 = val_metrics.box.ap50[i]

    val_table.append([name, p, r, ap50])

    print(f"{name:15s} | P: {p:.2f} | R: {r:.2f} | mAP50: {ap50:.2f}")


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,131,389 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 26.3±27.7 MB/s, size: 56.3 KB)
val: Scanning /content/cvcep-3/valid/labels.cache... 235 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 235/235 89.6Mit/s 0.0s
val: /content/cvcep-3/valid/images/WhatsApp-Image-2026-03-16-at-21-26-08-1-_jpeg.rf.c9b4faf0643d9194c7045c40c1e8c3a5.jpg: 1 duplicate labels removed
val: /content/cvcep-3/valid/images/WhatsApp-Image-2026-03-16-at-22-32-09-2-_jpeg.rf.c171b47b894648faa75eb5c5fe876d23.jpg: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 612, len(boxes) = 961. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Bo

In [16]:
# TEST METRICS
test_metrics = model.val(data=data_yaml, split="test")

print(" TEST RESULTS ")

print(f"mAP@0.5       : {test_metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {test_metrics.box.map:.3f}")
print(f"Precision     : {test_metrics.box.mp:.3f}")
print(f"Recall        : {test_metrics.box.mr:.3f}")

print("\n Per-Class Test Performance:")

test_table = []

for i in range(len(names)):
    name = names[i]
    p = test_metrics.box.p[i]
    r = test_metrics.box.r[i]
    ap50 = test_metrics.box.ap50[i]

    test_table.append([name, p, r, ap50])

    print(f"{name:15s} | P: {p:.2f} | R: {r:.2f} | mAP50: {ap50:.2f}")


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1148.0±332.7 MB/s, size: 40.2 KB)
val: Scanning /content/cvcep-3/test/labels.cache... 253 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 253/253 66.3Mit/s 0.0s
val: /content/cvcep-3/test/images/pole96_jpg.rf.2ca4233c107c7b0a685295b5d9a35538.jpg: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 502, len(boxes) = 792. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.7it/s 6.0s
                   all        253        792      0.613      0.423      0.432      0.269
                 bench         14         40      0.757      0.467      0.475      0.287
            

IndexError: index 14 is out of bounds for axis 0 with size 14

In [18]:

# 4) SAVE MODEL FOR FASTAPI (LOCAL COPY)
fastapi_model_path = "/content/best_blind_assist1.pt"

shutil.copy(best_path, fastapi_model_path)

print("\n MODEL SAVED FOR FASTAPI")

print("Local path:", os.path.abspath(fastapi_model_path))



 MODEL SAVED FOR FASTAPI
Local path: /content/best_blind_assist1.pt


In [19]:

# 5) EXPORT ONNX (OPTIMIZED INFERENCE)
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    simplify=True
)

print("\n DEPLOYMENT EXPORT COMPLETE")
print("\nONNX Path:")
print(onnx_path)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/MyDrive/FYP/blind_assist_v2_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 19, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 292ms
Prepared 4 packages in 17.63s
Installed 4 packages in 1.51s
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.25.0
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 20.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 23.4s, saved as '/

In [20]:
# 6) OPTIONAL: SAVE METRICS AS TABLE (FOR REPORT)
import pandas as pd
val_df = pd.DataFrame(val_table, columns=["Class", "Precision", "Recall", "mAP50"])
test_df = pd.DataFrame(test_table, columns=["Class", "Precision", "Recall", "mAP50"])

val_df.to_csv("/content/validation_metrics.csv", index=False)
val_df.to_csv("/content/drive/MyDrive/FYP/validation2_results.csv", index=False)
test_df.to_csv("/content/test_metrics.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/FYP/test2_results.csv", index=False)

print("\n Metrics saved as CSV files")



 Metrics saved as CSV files
